# 09 - Production API Ingestion Pipeline

**The 5 Pillars of Enterprise REST Ingestion:**
1. **Robust Pagination:** Walking offset/cursor pages until the API signals completion.
2. **Resilient Retries with Bounded Backoff:** Handling transient errors (429, 5xx, timeouts) vs failing fast on permanent errors (401, 403, 404).
3. **Validation & Dead-Letter Quarantine:** One dirty record must never fail an entire batch; clean records proceed, bad records land in dead-letter storage.
4. **Raw Replayability:** Landing raw immutable JSONL files before applying any transformations.
5. **Idempotency & Watermarking:** Ensuring pipeline replays are safe no-ops, and driving increments with `updated_at > watermark`.


In [ ]:
import sys, pathlib, os, json, time, re
import pandas as pd
import requests

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)
from pyspark.sql import functions as F, Window
from fake_api import ClaimsAPI

spark = get_spark("09-api-ingestion")
dbutils = get_dbutils()
print("Setup complete. Local API fixture and Spark session ready.")


---
### Drill 1: The "Single-Page" Trap

**Business Scenario:** Calling `GET /claims?page=1&limit=500`.
* **The Trap:** An HTTP 200 with 500 rows looks like complete success, but it is only ~4% of the total dataset ($12,061$ total records).
* **Lesson:** Always inspect response pagination metadata (`total`, `next_page`, `has_more`) before processing.


In [ ]:
api = ClaimsAPI()
response = api.get(page=1, limit=500)
payload = response.json()

page_rows = len(payload["data"])
total_rows = payload["total"]
coverage_pct = (page_rows / total_rows) * 100

print(f"HTTP Status:       {response.status_code}")
print(f"Rows on Page 1:    {page_rows:,}")
print(f"Total Rows at API: {total_rows:,}")
print(f"Page 1 Coverage:   {coverage_pct:.1f}% (96% of data is still uncollected!)")


---
### Drill 2: Robust Pagination Loop

**Goal:** Paginate until the API returns an empty page, collecting all records while logging progress per page.
* **Pagination Pattern:** Stop when `len(batch) == 0`, not merely when reaching `total` (protects against concurrent writes adding rows mid-extraction).
* **Expected Output:** 25 pages of 500 rows = $12,061$ records landed.


In [ ]:
# YOUR TURN:
# Build fetch_all_pages(api, page_limit=500)
# Loop pages until an empty batch is returned, tracking running totals.


In [ ]:
# SOLUTION - Drill 2
def fetch_all_pages(api, page_limit=500, since=None, quiet=False):
    """Pages through the REST API until an empty batch is returned."""
    all_records = []
    page = 1
    
    while True:
        resp = api.get(page=page, limit=page_limit, since=since, timeout=30)
        batch = resp.json().get("data", [])
        
        if not batch:
            break
            
        all_records.extend(batch)
        if not quiet:
            print(f"  Page {page:>2} -> {len(batch):>3} rows | Running Total: {len(all_records):>6,}")
            
        page += 1
        
    return all_records

api = ClaimsAPI()
records = fetch_all_pages(api, page_limit=500)
print()
print(f"API Transport calls: {api.calls:,} | Records landed: {len(records):,}")


---
### Drill 3: Bounded Exponential Backoff & Retry Logic

**Resiliency Rule:**
* **Retryable Errors (Transient):** `429` (Rate Limited), `500`, `502`, `503`, `504`, and socket `Timeout`.
* **Non-Retryable Errors (Permanent):** `400` (Bad Request), `401` (Unauthorized), `403` (Forbidden), `404` (Not Found), `422` (Unprocessable). **Fail fast and alert; never retry!**
* **Backoff Strategy:** Exponential delay `delay = base_delay * (2 ** attempt)` with a hard attempt limit.


In [ ]:
# YOUR TURN:
# Write get_with_retry(api, max_attempts=3, **kwargs)
# Catch transient HTTP errors/timeouts and backoff; fail fast on 4xx.


In [ ]:
# SOLUTION - Drill 3
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}
retry_audit_log = []

def get_with_retry(api, max_attempts=3, base_delay=0.01, **kwargs):
    """Executes HTTP request with bounded exponential backoff on transient errors."""
    for attempt in range(max_attempts):
        try:
            resp = api.get(timeout=30, **kwargs)
            
            # Check for retryable HTTP status codes
            if resp.status_code in RETRYABLE_STATUS_CODES:
                raise requests.exceptions.HTTPError(f"HTTP {resp.status_code} Transient", response=resp)
                
            resp.raise_for_status()
            return resp
            
        except requests.exceptions.Timeout:
            retry_audit_log.append({"page": kwargs.get("page"), "attempt": attempt + 1, "reason": "Timeout"})
        except requests.exceptions.HTTPError as exc:
            status = exc.response.status_code if exc.response is not None else None
            if status not in RETRYABLE_STATUS_CODES:
                raise  # Fail fast on 401/403/404!
            retry_audit_log.append({"page": kwargs.get("page"), "attempt": attempt + 1, "reason": f"HTTP {status}"})
            
        # Exponential backoff
        time.sleep(base_delay * (2 ** attempt))
        
    raise RuntimeError(f"Request failed after {max_attempts} attempts: {kwargs}")

# Test against a flaky API (fails every 7th call, times out every 11th call)
flaky_api = ClaimsAPI(fail_every=7, timeout_every=11)
records = []
page = 1

while True:
    res = get_with_retry(flaky_api, page=page, limit=500)
    batch = res.json().get("data", [])
    if not batch:
        break
    records.extend(batch)
    page += 1

print(f"Records landed:    {len(records):,}")
print(f"API calls made:    {flaky_api.calls}")
print(f"Failures injected: {flaky_api.failures_injected}")
print(f"Retries logged:    {len(retry_audit_log)}")
print()
print("Sample retry audit log:")
print(pd.DataFrame(retry_audit_log).head(6).to_string(index=False))


In [ ]:
# Verify Fail-Fast Behavior (401 Unauthorized must NEVER retry)
class MockUnauthorizedTransport:
    def __init__(self):
        self.calls = 0
    def get(self, **kwargs):
        self.calls += 1
        resp = requests.Response()
        resp.status_code = 401
        raise requests.exceptions.HTTPError("401 Unauthorized", response=resp)

blocked_client = MockUnauthorizedTransport()
try:
    get_with_retry(blocked_client, page=1)
except requests.exceptions.HTTPError as exc:
    print(f"Correctly failed fast on {exc} after {blocked_client.calls} call (Zero wasted retries!)")


---
### Drill 4: Data Quality Validation & Dead-Letter Quarantine

**The Core Rule:** A single corrupted record in an API payload must **never crash the entire batch pipeline**.
* Clean records are parsed and written to the pipeline.
* Bad records are diverted into a **Dead-Letter Queue (Quarantine)** with the exact failure reason recorded.
* **Validation Rules:**
  1. Required fields present: `claim_id`, `member_id`, `provider_id`, `claim_date`, `status`, `billed_amount`.
  2. Date matches `YYYY-MM-DD`.
  3. Status in domain: `Submitted`, `Approved`, `Denied`.
  4. Amounts are numeric or cleanable currency strings (e.g. `$1,250.50`).


In [ ]:
# YOUR TURN:
# Implement validate_record(record) -> (clean_record, reasons)
# Implement split_records(records) -> (valid_list, rejected_list)


In [ ]:
# SOLUTION - Drill 4
REQUIRED_FIELDS = ["claim_id", "member_id", "provider_id", "claim_date", "status", "billed_amount"]
DATE_REGEX = re.compile(r"^\d{4}-\d{2}-\d{2}$")
VALID_STATUSES = {"SUBMITTED", "APPROVED", "DENIED"}
MONEY_REGEX = re.compile(r"^\$?\s*-?[\d,]+(\.\d{1,2})?$")

def parse_currency(val):
    """Converts currency string or number to float."""
    if val is None or isinstance(val, bool):
        return None, False
    if isinstance(val, (int, float)):
        return float(val), False
    if isinstance(val, str) and MONEY_REGEX.match(val.strip()):
        cleaned = val.strip().lstrip("$").replace(",", "")
        return float(cleaned), True
    return None, False

def validate_record(record):
    reasons = [f"missing_{field}" for field in REQUIRED_FIELDS if record.get(field) in (None, "")]
    
    # Date validation
    if "missing_claim_date" not in reasons and not DATE_REGEX.match(str(record.get("claim_date", ""))):
        reasons.append("invalid_claim_date")
        
    # Status validation
    if "missing_status" not in reasons and str(record.get("status", "")).strip().upper() not in VALID_STATUSES:
        reasons.append("invalid_status")
        
    # Billed amount validation
    billed, coerced_billed = parse_currency(record.get("billed_amount"))
    if "missing_billed_amount" not in reasons and billed is None:
        reasons.append("invalid_billed_amount")
        
    paid, coerced_paid = parse_currency(record.get("paid_amount"))
    if record.get("paid_amount") not in (None, "") and paid is None:
        reasons.append("invalid_paid_amount")
        
    clean_rec = dict(record)
    if not reasons:
        clean_rec["status"] = str(clean_rec["status"]).strip().title()
        clean_rec["billed_amount"] = billed
        clean_rec["paid_amount"] = paid
        clean_rec["_coerced_amounts"] = coerced_billed or coerced_paid
        
    return clean_rec, reasons

def split_records(all_records):
    valid, rejected = [], []
    for rec in all_records:
        clean_rec, reasons = validate_record(rec)
        if reasons:
            rejected.append({"claim_id": rec.get("claim_id"), "reasons": ", ".join(reasons)})
        else:
            valid.append(clean_rec)
    return valid, rejected

valid_records, rejected_records = split_records(records)
print(f"Valid clean records:     {len(valid_records):,}")
print(f"Quarantined records:     {len(rejected_records):,}")
print(f"Coerced currency amounts: {sum(r['_coerced_amounts'] for r in valid_records):,}")
print()
print("Dead-Letter Quarantined Samples:")
print(pd.DataFrame(rejected_records))


---
### Drill 5: Landing Raw Immutable Files & Writing Typed Bronze Parquet

**Medallion Storage Best Practice:**
1. **Landing Zone (`landing/claims_raw.jsonl`):** Store raw API payloads exactly as received (defects intact). If business logic changes, you replay the raw bytes without re-calling the third-party API.
2. **Dead-Letter Zone (`landing/claims_rejected.jsonl`):** Store quarantined records with error annotations.
3. **Bronze Layer (`bronze/claims.parquet`):** Validated, typed columnar dataset for high-speed downstream processing.


In [ ]:
scratch_dir = ROOT / "data" / "scratch"
landing_dir = scratch_dir / "landing"
bronze_dir = scratch_dir / "bronze"
landing_dir.mkdir(parents=True, exist_ok=True)
bronze_dir.mkdir(parents=True, exist_ok=True)

raw_path = landing_dir / "claims_raw.jsonl"
reject_path = landing_dir / "claims_rejected.jsonl"
bronze_path = bronze_dir / "claims.parquet"

# 1. Land Raw JSONL (Immutable landing zone)
with raw_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")

# 2. Land Dead-Letter Queue
with reject_path.open("w", encoding="utf-8") as f:
    for rec in rejected_records:
        f.write(json.dumps(rec) + "\n")

# 3. Cast to typed Bronze Parquet
df_bronze = pd.DataFrame(valid_records)
for date_col in ["claim_date", "submitted_date", "updated_at", "ingested_at"]:
    df_bronze[date_col] = pd.to_datetime(df_bronze[date_col], errors="coerce")
    
df_bronze["billed_amount"] = df_bronze["billed_amount"].astype(float)
df_bronze["paid_amount"] = pd.to_numeric(df_bronze["paid_amount"], errors="coerce")
df_bronze.to_parquet(bronze_path, index=False)

print(f"Raw Landing:  {raw_path.name}  ({raw_path.stat().st_size / 1e6:.2f} MB, {len(records):,} lines)")
print(f"Dead Letter:  {reject_path.name} ({len(rejected_records)} quarantined rows)")
print(f"Bronze Delta: {bronze_path.name} ({bronze_path.stat().st_size / 1e6:.2f} MB, {len(df_bronze):,} rows)")


In [ ]:
# Querying the freshly landed Bronze dataset in Spark
is_remote = spark.__module__.startswith("pyspark.sql.connect") or hasattr(spark, "client")
if is_remote:
    # Databricks Connect session: transfer via in-memory pandas frame
    bronze_spark = spark.createDataFrame(df_bronze.head(500))
    print(f"PySpark DataFrame created (Connect session): {bronze_spark.count():,} rows")
else:
    bronze_spark = spark.read.parquet(str(bronze_path))
    print(f"PySpark read from Bronze Parquet: {bronze_spark.count():,} rows")

bronze_spark.select("claim_id", "claim_date", "status", "billed_amount", "paid_amount").show(3, truncate=False)


---
### Drill 6: Incremental Extraction with High-Watermarks

**Goal:** Only query the API for records modified since the last batch.
* **Deriving Watermark:** `max_updated_at = bronze_frame["updated_at"].max()`.
* **Passing to API:** `GET /claims?since={watermark}`.
* **Expected Result:** `0` new records $\to$ A clean, successful no-op (not an error).
* **Lookback Buffer:** Always apply an overlap window (`since = watermark - 7 days`) to catch delayed/late-arriving records.


In [ ]:
# 1. Derive watermark from landed batch
watermark = df_bronze["updated_at"].max()
print(f"Landed Batch High-Watermark: {watermark}")

# 2. Query API with watermark
incremental_api = ClaimsAPI()
new_deltas = fetch_all_pages(incremental_api, since=str(watermark), quiet=True)
print(f"Records updated after watermark: {len(new_deltas)} (Successful no-op)")

# 3. Lookback buffer (e.g. 7-day lookback window)
lookback_date = str(watermark - pd.Timedelta(days=7))
catchup_records = [r for r in records if str(r.get("updated_at", "")) > lookback_date]
print(f"Records in 7-day lookback window: {len(catchup_records):,} (Safely captured by lookback)")


---
### Drill 7: Idempotency & Replay Verification

**The Production Golden Rule:** Re-running the pipeline on the same day/batch must produce the **exact same row count without duplicates**.
* **Replay-safe:** Writing to the partition/path overwrites or upserts cleanly on the business key (`claim_id`).
* **Non-idempotent Anti-Pattern:** Blind `APPEND` causes duplicate rows, inflated KPIs, and distorted billing totals.


In [ ]:
# Re-run full ingestion and overwrite Bronze
rerun_api = ClaimsAPI()
rerun_records = fetch_all_pages(rerun_api, quiet=True)
valid_rerun, _ = split_records(rerun_records)

rows_before = len(pd.read_parquet(bronze_path))
df_rerun = pd.DataFrame(valid_rerun)
for date_col in ["claim_date", "submitted_date", "updated_at", "ingested_at"]:
    df_rerun[date_col] = pd.to_datetime(df_rerun[date_col], errors="coerce")
df_rerun["billed_amount"] = df_rerun["billed_amount"].astype(float)
df_rerun["paid_amount"] = pd.to_numeric(df_rerun["paid_amount"], errors="coerce")
df_rerun.to_parquet(bronze_path, index=False)

rows_after = len(pd.read_parquet(bronze_path))
distinct_claims = pd.read_parquet(bronze_path)["claim_id"].nunique()

is_idempotent = (rows_before == rows_after)
print(f"Rows before re-run:       {rows_before:,}")
print(f"Rows after re-run:        {rows_after:,}")
print(f"Distinct claim IDs:       {distinct_claims:,} (Stable across replays)")
print(f"Replay-safe / Idempotent: {is_idempotent} (Zero surplus duplicate rows added)")


---
### Drill 8: Warehouse Impact of Non-Idempotent Ingestion

**Production Post-Mortem:**
In historical tables (`bronze.claims`), an unmitigated re-run occurred on `2025-04-16T02:05:00Z` (302 duplicate rows).
Here we query DuckDB to detect and audit the duplication surplus.


In [ ]:
# Warehouse Analysis: Diagnosing non-idempotent replays in historical data
print("--- Warehouse Replay Audit (DuckDB) ---")
print(q("""
SELECT 
    COUNT(*) AS bronze_rows, 
    COUNT(DISTINCT claim_id) AS distinct_claim_ids,
    COUNT(*) - COUNT(DISTINCT claim_id) AS surplus_rows,
    (SELECT COUNT(*) FROM (SELECT claim_id FROM bronze.claims GROUP BY 1 HAVING COUNT(*) > 1)) AS duplicated_claim_ids
FROM bronze.claims
""").to_string(index=False))

print()
print("--- Ingestion Batch Breakdown ---")
print(q("""
SELECT 
    _ingest_batch, 
    COUNT(*) AS rows_loaded, 
    MIN(claim_date) AS first_claim, 
    MAX(claim_date) AS last_claim
FROM bronze.claims 
GROUP BY 1 
ORDER BY rows_loaded DESC
""").to_string(index=False))

print()
print("--- Replay Contamination in April 2025 ---")
print(q("""
SELECT 
    date_trunc('month', claim_date) AS claim_month,
    COUNT(*) AS bronze_rows,
    COUNT(*) FILTER (WHERE _ingest_batch = '2025-04-16T02:05:00Z') AS replay_rows
FROM bronze.claims
WHERE claim_date >= DATE '2025-03-01' AND claim_date < DATE '2025-05-01'
GROUP BY 1 
ORDER BY 1
""").to_string(index=False))

silver_april = q1("SELECT COUNT(*) FROM silver.claims WHERE date_trunc('month', claim_date) = DATE '2025-04-01'")
print(f"Silver April clean deduplicated rows: {silver_april:,}")


---
### The 6-Point Senior DE Interview Answer on REST Ingestion

When an interviewer asks: *"How do you design a robust, enterprise-grade REST API ingestion pipeline?"*

1. **Authentication:** Use OAuth2 with cached tokens; automatically refresh token on `401 Unauthorized` without entering a retry loop.
2. **Pagination:** Walk pages using an offset/cursor until receiving an empty batch or null next-token. Persist the last cursor to allow resumption after crashes.
3. **Resilience & Backoff:** Implement bounded exponential backoff (e.g. 3 attempts) with jitter for transient errors (`429`, `5xx`, timeouts); fail fast on permanent `4xx` client errors.
4. **Validation & Dead-Letter Queue:** Validate incoming records against schema contracts (required keys, date formats, enum domains). Route bad rows to a quarantine dead-letter store so they never block the pipeline.
5. **Raw Replayability & Idempotency:** Land raw payloads immutably (`landing/*.jsonl`) before parsing. Use atomic overwrites or `MERGE INTO` on primary keys to ensure replays are idempotent.
6. **Observability:** Track and publish run metrics: records extracted, pages fetched, retry count, quarantine rate, and pipeline latency. Alert on sudden spikes in rejection rates.
